# Study 819 — Abnormal-Volume Shock — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4083, 'spread_bps': 0.55, 't_nw': 0.54, 't_1s': 0.52, 'hi_bps': 7.44, 'lo_bps': 6.89, 'welch_t': 0.21, 'gross_sharpe': 0.13, 'placebo_obs': 0.55, 'placebo_mean': 0.015, 'placebo_sd': 0.865, 'placebo_p': 0.266, 'placebo_sigma_right': 0.62, 'placebo_draws': 1000, 'era_early_bps': -0.48, 'era_early_t': -0.4, 'era_early_n': 1949, 'era_late_bps': 1.49, 'era_late_t': 0.94, 'era_late_n': 2134, 'timer_1_gross': 0.55, 'timer_1_cost': 2.14, 'timer_1_net': -1.59, 'timer_1_t': -1.49, 'timer_5_gross': 0.55, 'timer_5_cost': 10.14, 'timer_5_net': -9.59, 'timer_5_t': -8.98, 'null_mean_t': -0.27, 'null_sd_t': 1.05, 'null_fire': 2, 'planted_t': 21.64, 'planted_welch': 22.46}

## The headline — long-high-avol / short-low-avol spread

Daily equal-weight top-30% minus bottom-30% abnormal-volume spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-avol {R['hi_bps']:+.2f} vs low-avol {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : +0.55 bps/day  NW(10) t = +0.54  one-sample t = +0.52
books         : high-avol +7.44 vs low-avol +6.89 bps (Welch t = +0.21)
gross Sharpe  : 0.13 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.5f}  "
      f"(~{R['placebo_sigma_right']:.2f} sigma into the right tail)")

observed +0.55 bps vs placebo mean +0.015 (sd 0.865) -> p = 0.26600  (~0.62 sigma into the right tail)


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print('  -> the sign FLIPS between halves; neither is significant')

2010-2017 (n=1949): -0.48 bps  NW t = -0.40
2018-2026 (n=2134): +1.49 bps  NW t = +0.94
  -> the sign FLIPS between halves; neither is significant


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross +0.55 -> net -1.59 bps/day (cost 2.14/day, t=-1.49)
5 bps one-way: gross +0.55 -> net -9.59 bps/day (cost 10.14/day, t=-8.98)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from volume_shock import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=819+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0020, seed=819, n_assets=40, n_days=1500))
print(f"planted (edge=0.0020): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.40 (sd 1.32), |t|>=2 in 2/8


planted (edge=0.0020): NW t = +21.64, Welch t = +22.46


## Verdict

- **Signal — None.** The claimed Garfinkel-Sokobin abnormal-volume drift does **not** replicate as a tradable spread on 50 liquid US mega-caps: the long-high-avol / short-low-avol spread is **+0.55 bps/day** (NW *t* = **+0.54**) — correctly signed but statistically absent, only ~0.62σ into the placebo (p = 0.27), and it **flips sign** across the two eras (*t* = -0.40 / +0.94). The synthetic control recovers a *planted* relation overwhelmingly (*t* = +21.6; null ≈ N(0,1)), so the flat result is the data, not machinery.
- **Tradability — Mirage.** The right-signed +0.55 bps/day gross tilt is dwarfed by the 2.14 bps/day friction at 1 bp one-way, net **-1.59 bps/day** (*t* = -1.49); at 5 bps **-9.59 bps/day**.